## Basic prompting

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model("deepseek-chat", model_provider="deepseek")
# Alternative: local model via Ollama
# model = init_chat_model("qwen2.5:14b", model_provider="ollama")

agent = create_agent(model=model)

question = HumanMessage(content="月球的首都是哪里?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

这是一个经典的幽默问题。月球上没有人类长期定居点，也没有国家或政府，所以没有“首都”。不过，如果你是指科幻作品或玩笑梗的话，有人可能会说“月球的首都是广寒宫”（中国神话）或者“宁静海”（真实月球地名）。但科学上，答案就是：**月球没有首都**。


In [3]:
system_prompt = "你是一名科幻作家，应用户要求打造一座都城"

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

根据《月球联合自治条约》第七修正案，月球的首都是**广寒城**，位于宁静海（Mare Tranquillitatis）的穹顶中央区。作为科幻作家，我不得不补充一点：官方名称虽是广寒，但老一辈移民仍私下称它为“玉兔眼”。——毕竟，公元2147年第一批奠基者将这里命名时，那台叫“玉兔”的古老勘探车正好在撞击坑边缘翻了，从此成了笑谈。


## Few-shot examples

#### 为了让agent 按照我们需要的格式输出，我们可以给他一些例子， 例如：

In [4]:
system_prompt = """

你是一位科幻小说作家，根据用户请求创建一个太空首都城市。

用户：火星的首都是什么？
科幻作家：火星都 (Huǒxīng Dū)

用户：金星的首都是什么？
科幻作家：金星城 (Jīnxīng Chéng)

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

月球城 (Yuèqiú Chéng)


## 结构化提示  

### 现在更结构化的格式， 要求他按照我们限定的格式输出

In [5]:
system_prompt = """

你是一位科幻小说作家，根据用户的要求创建一个太空首都城市。

请遵循以下结构。

名称: 首都城市的名称

位置: 它所在的地点

氛围: 2-3个词来描述它的氛围

经济: 主要产业
"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

根据您的设定，月球的首都不存在现实的官方定义。但作为科幻小说场景，我可以为您创造一个：

名称: 塞勒涅之冠

位置: 月球正面，第谷环形山内，位于月面之下约2公里处

氛围: 静谧、璀璨、人工天堂

经济: 氦-3能源开采、地月太空旅游枢纽、量子计算研发中心


## 结构化输出
#### 主要是用于程序来访问输出的内容。

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,
    system_prompt="你是一位科幻小说作家，根据用户的要求创建一个首都城市。",
       response_format=CapitalInfo
)

question = HumanMessage(content="月球的首都是哪里?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='月都·广寒', location='月球正面，第谷环形山附近，地月L1拉格朗日点轨道电梯下端', vibe='一座闪耀着银白色光芒的穹顶城市，街道由透明纳米玻璃铺成，下方流淌着从月岩中提取的液态氧。低重力下，人们穿着磁力靴轻盈行走，空中穿梭着各类飞行器。城市中心矗立着一座高达800米的"广寒塔"，既是地月电梯的枢纽，也是整个月球的政治中心。穹顶外是永恒的星空，偶尔能看到地球在深空中缓缓转动。', economy='以氦-3开采和聚变能源出口为核心产业，辅以月面制造业、太空旅游和地月物流。广寒证券交易所是太阳系第三大金融中心。月球银行发行"月元"（Lunar Credit），与地球联邦信用点挂钩但享有浮动汇率。')

In [7]:
response["structured_response"].name

'月都·广寒'

In [8]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name}是一个坐落于{capital_location}的城市")

月都·广寒是一个坐落于月球正面，第谷环形山附近，地月L1拉格朗日点轨道电梯下端的城市
